# Project SD-04 — Email Threads RAG

> Goal: Turn email — the raw MIME blobs and reply chains of an `.eml` — into a
> RAG corpus by parsing it with the Python stdlib `email` package: one
> `Document` per message, quoted blocks stripped, and `sender` / `date` /
> `thread_id` / `depth` metadata attached so retrieval can answer
> thread-aware questions.

```
Loader      : stdlib email (BytesParser + policy.default) — one Document per message
Splitter    : RecursiveCharacterTextSplitter (per message body)
Embedding   : Gemini Embedding
Vector DB   : Chroma
Retriever   : Similarity Search (Top-K) + metadata filters
Prompt      : Context + Question (with sender/date stamps)
LLM         : Gemini 2.5 Flash
```

Learn:

* Parsing is the changed block: raw MIME → structured message
* Reply-chain diseases: quoted-text duplication, attribution, ordering
* Thread reconstruction via `In-Reply-To` / `References`


## The naive way (what breaks)

Email arrives as one opaque MIME blob — a *container*, not text. The tempting
shortcut is to read the file as one big string and hand it straight to the text
splitter:

```python
raw = open("message.eml", encoding="utf-8", errors="replace").read()
chunks = splitter.split_text(raw)   # naive: split the blob itself
```

Four things break — and email makes all of them worse than ordinary documents.

1. **The blob is mostly not prose.** A single message is headers + MIME
   boundaries + a base64 image + a signature. The real body is a few dozen
   characters buried inside thousands of container bytes; embedding the whole
   blob dilutes every vector with base64 noise.
2. **Quoted-text duplication — `n(n+1)/2` bloat.** In a reply chain every
   message re-quotes the history. Message 1 holds 1 copy of the original,
   message 2 holds 2, message 3 holds 3 … a thread of `n` messages stores the
   original ~`n(n+1)/2` times. The index fills with duplicates of one sentence.
3. **Attribution collapse (~40% wrong speaker).** Once quoted blocks are glued
   into a flat chunk, the model cannot tell who wrote which line — studies of
   email RAG find roughly 40% of speakers get misattributed inside quoted
   blocks ("Alice said" for text Bob actually wrote).
4. **No temporal ordering.** A flat chunk hides which message came first. The
   classic thread questions — *"what was the original proposal?"* (depth 0) and
   *"what did the latest message decide?"* (newest date) — become unanswerable.

The demo below shows failure 1 concretely on the real sample file.


In [ ]:
import os

EMAIL_SAMPLE = "../../Data/SD-04-email/raw_email_with_nested_attachment.eml"

if not os.path.exists(EMAIL_SAMPLE):
    print(f"Sample file not found:\n  {EMAIL_SAMPLE}\n"
          "Look for it under Data/SD-04-email/.")
else:
    print(f"Found sample: {EMAIL_SAMPLE}")


In [ ]:
if os.path.exists(EMAIL_SAMPLE):
    raw = open(EMAIL_SAMPLE, encoding="utf-8", errors="replace").read()
    print(f"raw MIME blob: {len(raw):,} characters")
    print(raw[:170])


In [ ]:
if os.path.exists(EMAIL_SAMPLE):
    raw = open(EMAIL_SAMPLE, encoding="utf-8", errors="replace").read()
    marker = "Here is a test of an attachment via email."
    body = raw[raw.find(marker):raw.find(marker) + len(marker)]
    print("REAL message body:", repr(body))
    print(f"blob {len(raw):,} chars vs body {len(body)} chars "
          f"-> ~{100 - (100 * len(body) // len(raw))}% container noise")
    print(f"MIME boundary markers in one message: {raw.count('--Apple-Mail')}")


## 0 · Setup — environment & keys

**WHAT:** Loads `.env` so the Gemini API key is available, checks it is set
(masked), and imports the whole RAG stack plus the stdlib email parser.

**WHY:** Email parsing is the changed block of this project, and the parser we
use is the Python standard library — `email.parser.BytesParser` with
`policy.default`. There is **no install cell**: `email`, `glob`, `os`, and `re`
ship with Python, and everything else matches the Project 04 stack.

**WHAT TO EXPECT:** `True` from `load_dotenv()` (a `.env` at the repo root with
`GOOGLE_API_KEY=...`), a masked key check that prints only the first four
characters, then all imports resolving cleanly.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

key = os.getenv("GOOGLE_API_KEY")
if key:
    print(f"GOOGLE_API_KEY set (starts with {key[:4]}…)")
else:
    print("No GOOGLE_API_KEY found — copy .env.example to .env and add yours.")


In [ ]:
import os
import re
import glob
from email import policy
from email.parser import BytesParser

from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document


## 1 · Load — parse with the stdlib, not a loader

**WHAT:** `BytesParser(policy=policy.default)` turns the raw bytes of an `.eml`
into an `email.message.EmailMessage`. Three tiny helpers then unpack it:
`parse_eml()` opens the file, `body_text()` walks the MIME part tree collecting
only the plain-text parts (skipping attachments, images, and signatures), and
`envelope()` reads the headers (`From`, `To`, `Date`, `Subject`, `Message-ID`,
`In-Reply-To`, `References`).

**WHY:** This is the changed block. The old community email-loader family is
archived and gone, and the naive flatten above showed why a raw blob is not
text: parsing is the *unpacking* step that separates real prose from base64
images, signatures, and MIME boundaries.

**WHAT TO EXPECT:** The envelope printed for the committed sample — a body of
42 real characters, not a 4,951-character blob.


In [ ]:
def parse_eml(path):
    """Read a .eml file into an EmailMessage (stdlib email parser)."""
    with open(path, "rb") as f:
        return BytesParser(policy=policy.default).parse(f)


In [ ]:
def body_text(msg):
    """Walk the MIME tree, keeping only the plain-text parts."""
    if msg.is_multipart():
        return "\n\n".join(body_text(p) for p in msg.iter_parts())
    if msg.get_content_type() == "text/plain":
        return msg.get_content()
    return ""


In [ ]:
def envelope(msg):
    """Envelope headers + cleaned plain-text body in one dict."""
    keys = ["From", "To", "Subject", "Date",
            "Message-ID", "In-Reply-To", "References"]
    head = {k: str(msg.get(k) or "") for k in keys}
    head["sender"] = head.pop("From") or "unknown"
    head["body"] = body_text(msg).strip()
    return head


In [ ]:
if os.path.exists(EMAIL_SAMPLE):
    fields = envelope(parse_eml(EMAIL_SAMPLE))
    print("From:   ", fields["sender"])
    print("To:     ", fields["to"])
    print("Date:   ", fields["date"])
    print("Subject:", fields["subject"])
    print("Body:   ", repr(fields["body"]))


**Thread reconstruction — `In-Reply-To` and `References`.** Real reply chains
carry two headers that encode the conversation tree:

* `Message-ID` — this message's own id.
* `In-Reply-To` — the id of the message this one directly answers.
* `References` — the full ancestor list, oldest first, ending at the parent.

`thread_id` is the root `Message-ID` (the first entry of `References`, or the
message's own id if it is the root). `depth` is the number of ancestors in
`References` — how many messages precede this one in the chain. The committed
sample is a single message with no reply headers, so it is its own root at
`depth 0`; the Enron inbox (when present) is a real employee thread with proper
headers.


In [ ]:
def thread_info(fields):
    """thread_id = root message id; depth = number of ancestors."""
    refs = re.findall(r"<[^>]+>", fields["references"])
    thread_id = refs[0] if refs else fields["message_id"]
    depth = len(refs)
    return thread_id, depth


**Strip the quoted blocks.** Reply chains are unreadable unless the quoted
history is removed. The classic markers are `On <date>, <person> wrote:` lines
and every `>`-prefixed line. `strip_quotes()` drops those blocks, and
`dedup_repeated()` drops repeated long lines left over from nested forwards —
the mechanical counterpart of the `n(n+1)/2` bloat from the naive way. Both are
simple heuristics; production parsing adds `email.utils` date handling and
per-mail-client quoted-text markers, but the idea is the same — **keep the new
text, drop the re-quoted history.**


In [ ]:
def strip_quotes(text):
    """Drop 'On ... wrote:' blocks and '>' prefixed quoted lines."""
    lines, in_quote = [], False
    for line in text.splitlines():
        if re.match(r"^On .+ wrote:", line.strip()) or line.strip().startswith(">"):
            in_quote = True
            continue
        if in_quote and line.strip() == "":
            in_quote = False
        if not in_quote:
            lines.append(line)
    return "\n".join(lines).strip()


In [ ]:
def dedup_repeated(text, min_len=40):
    """Drop repeated long lines left by nested forwards."""
    out, seen = [], set()
    for line in text.splitlines():
        key = line.strip().lower()
        if len(key) >= min_len and key in seen:
            continue
        if len(key) >= min_len:
            seen.add(key)
        out.append(line)
    return "\n".join(out)


In [ ]:
snippet = (
    "On Feb 22, 2007, at 11:20 AM, Alice wrote:\n"
    "> Here is the plan.\n"
    "> Let me know.\n"
    "\n"
    "Sounds good, will do.\n"
)
print("after strip_quotes:", repr(strip_quotes(snippet)))


**Enron extension — a real thread, when present.** This repo's `.gitignore`
excludes a small Enron corpus at
`Data/SD-04-email/enron/allen-p/inbox/` — one employee's inbox of
real `.eml` messages. It is *optional*: the flag cell computes
`ENRON_AVAILABLE` straight from the filesystem with `os.path.isdir` and
`glob`, and the notebook behaves correctly with or without it. The committed
sample is always loaded; Enron is loaded only when the files exist.


In [ ]:
ENRON_DIR = "../../Data/SD-04-email/enron/allen-p/inbox"
enron_files = sorted(glob.glob(os.path.join(ENRON_DIR, "*.eml")))

ENRON_AVAILABLE = os.path.isdir(ENRON_DIR) and len(enron_files) > 0

print("ENRON_AVAILABLE =", ENRON_AVAILABLE)
if ENRON_AVAILABLE:
    print(f"found {len(enron_files)} Enron emails in allen-p/inbox")
else:
    print("no Enron inbox present — using the committed sample only")


In [ ]:
def eml_to_document(path):
    """One message -> one Document: cleaned body + thread metadata."""
    fields = envelope(parse_eml(path))
    thread_id, depth = thread_info(fields)
    body = dedup_repeated(strip_quotes(fields["body"]))

    content = f"Subject: {fields['subject']}\n{body}"
    metadata = {
        "source": path,
        "sender": fields["sender"],
        "date": fields["date"],
        "thread_id": thread_id,
        "depth": depth,
    }
    return Document(page_content=content, metadata=metadata)


In [ ]:
docs = []
if os.path.exists(EMAIL_SAMPLE):
    docs.append(eml_to_document(EMAIL_SAMPLE))
if ENRON_AVAILABLE:
    docs.extend(eml_to_document(p) for p in enron_files)

print(f"parsed {len(docs)} message(s)")
if ENRON_AVAILABLE:
    print(f"  {len(enron_files)} Enron + 1 committed sample")


In [ ]:
for d in docs[:5]:
    m = d.metadata
    print(f"[{m['sender']}] {m['date']} depth={m['depth']}")
    print(f"   {d.page_content[:70]!r}")


## 2 · Split — per message, metadata inherited

**WHAT:** `RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)`
slices each message body into chunks. A small `chunk_size` suits email: bodies
are short, and a 500-character window keeps one train of thought intact.

**WHY:** `split_documents()` copies each `Document`'s metadata onto every
chunk it produces, so `sender` / `date` / `thread_id` / `depth` survive into
the index. That is what later lets retrieval *filter* by speaker, thread, or
recency — not just rank by similarity.

**WHAT TO EXPECT:** One chunk per message for the committed sample (a short
body does not split), and many more chunks when the Enron inbox is present —
each carrying its message's metadata.


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
chunks = splitter.split_documents(docs)

print(f"{len(docs)} message(s) -> {len(chunks)} chunks")


In [ ]:
for c in chunks[:3]:
    m = c.metadata
    print(f"thread={m['thread_id'][:24]}... depth={m['depth']} "
          f"sender={m['sender']}")
    print(f"   {c.page_content[:60]!r}")


## 3 · Embed — text → vectors

**WHAT:** `GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")`
turns every chunk into a numeric vector; similar content lands close together
in vector space.

**WHY:** Because the naive blob was full of base64 noise, its vectors encoded
junk. Parsed chunks embed clean prose, so "what was the original message
about" can match the actual sentence.

**WHAT TO EXPECT:** An embeddings object, then the dimensionality of one
embedded query.


In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

sample_vec = embeddings.embed_query("What is this email about?")
print(len(sample_vec), "dimensions per chunk")


## 4 · Store — index the vectors in Chroma

**WHAT:** `Chroma.from_documents(documents=chunks, embedding=embeddings)`
embeds every chunk and writes it into a Chroma collection; a
`chroma_langchain_db/` folder appears next to the notebook.

**WHY:** The vector store is the pipeline's memory. Crucially, the chunk
metadata — `sender`, `date`, `thread_id`, `depth` — is indexed alongside the
vectors, so retrieval can *filter* on it instead of only ranking by
similarity.

**WHAT TO EXPECT:** A `Chroma` object (and the folder on disk).


In [ ]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
)


## 5 · Retrieve — top-k, with thread metadata

**WHAT:** `similarity_search(query, k=3)` embeds the question and returns the
three closest chunks. We stamp each hit with its sender and date and join them
into a `context` string; a second cell shows a `depth=0` filter that pins the
*original proposal*.

**WHY:** Two of the four naive failures are fixed right here. Sender/date
stamps in the context fix **attribution** ("who said this?") and **ordering**
("which message came first?"), and the `depth` metadata filter answers the
"original proposal" question precisely.

**WHAT TO EXPECT:** Three chunks whose metadata tells you who wrote them and
when.


In [ ]:
query = "What was the original message about?"
retrieved = vector_store.similarity_search(query, k=3)

print(f"retrieved {len(retrieved)} chunks for: {query!r}")


In [ ]:
def fmt_doc(d):
    m = d.metadata
    return (f"[From {m['sender']} | {m['date']} | depth {m['depth']}]\n"
            f"{d.page_content}")

context = "\n\n".join(fmt_doc(d) for d in retrieved)

for d in retrieved:
    print(f"[{d.metadata['sender']}] {d.metadata['date']} "
          f"-> {d.page_content[:60]!r}")


In [ ]:
original = vector_store.similarity_search(
    "original proposal",
    k=3,
    filter={"depth": 0},
)
print(f"depth=0 (root message) hits: {len(original)}")
for d in original:
    print(f"  [{d.metadata['sender']}] {d.page_content[:60]!r}")


## 6 · Prompt — package context + question

**WHAT:** A `ChatPromptTemplate` wraps "answer using ONLY the provided
context" (with an explicit "I don't know" fallback) around the `context` and
`question` slots. The context already carries `[From … | date | depth]`
stamps.

**WHY:** The prompt is the contract that stops hallucination. For email it also
relays the speaker/time stamps: because the context says *who* wrote *what*
*when*, the model can answer the two thread questions the naive pipeline
couldn't.

**WHAT TO EXPECT:** A `ChatPromptTemplate`, then the rendered `messages`.


In [ ]:
template = """
You are a helpful assistant.

Answer the question using ONLY the provided context.
If the answer is not contained in the context, say:
"I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)


In [ ]:
messages = prompt.invoke({"context": context, "question": query})
print(messages)


## 7 · Answer — the LLM reads the prompt

**WHAT:** `ChatGoogleGenerativeAI(model="gemini-2.5-flash")` invokes the filled
`messages`; the answer is printed.

**WHY:** The final block. The model reads the stamped evidence plus the
question and answers — grounded in the parsed email, not in its own memory.

**WHAT TO EXPECT:** A short answer naming the sender (Jamis Buck) and the
message's purpose — "Testing attachments", a test of sending an attachment via
email.


In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
response = llm.invoke(messages)
print(response.content)


## 8 · Try it yourself — your sandbox

The `ask()` helper runs retrieve → stamp → prompt → answer in one call. The
two classic thread questions — "what was the original proposal?" and "what did
the latest message decide?" — are pre-adapted to this single-message sample.
Change `k`, add a `filter`, or reword the questions and re-run.


In [ ]:
def ask(question, k=3, **filters):
    hits = vector_store.similarity_search(question, k=k, filter=filters or None)
    ctx = "\n\n".join(fmt_doc(d) for d in hits)
    return llm.invoke(prompt.invoke({"context": ctx, "question": question})).content


In [ ]:
print(ask("Who sent this email and what did they say about testing attachments?"))


In [ ]:
print(ask("When was this email sent and what was its subject?", k=2))


## What you should notice

* **Parsing, not loaders, is the changed block.** The pipeline is identical to
  Project 04; the difference is that an `.eml` becomes text through the stdlib
  `email` parser instead of a convenience loader.
* **A MIME blob is a container, not text.** The committed sample is ~4,950
  characters; the real body is 42. Naive flattening would embed base64 images
  and signatures as if they were prose.
* **Reply chains bloat quadratically.** Each message re-quotes history, so a
  thread of `n` messages stores the original ~`n(n+1)/2` times — strip quotes
  and dedup forwards *before* indexing.
* **Attribution needs sender stamps.** Roughly 40% of speakers are
  misattributed inside quoted blocks; stamping each chunk with its sender fixes
  the "who said that?" question.
* **Ordering lives in metadata.** `date` and `depth` turn "the latest message"
  and "the original proposal" from unanswerable into filterable questions.
* **`In-Reply-To` / `References` rebuild the tree.** `thread_id` (root id) and
  `depth` (ancestor count) reconstruct the conversation from a flat inbox.


## Exercises

1. **Simulate a reply chain.** Write a tiny `.eml` with a quoted `On … wrote:`
   block plus `In-Reply-To` / `References` headers, load it, and compare the
   chunk count and content with and without `strip_quotes()`.
2. **Answer the "latest message" with a filter.** Parse the `date` strings
   with `email.utils.parsedate_to_datetime`, add a Chroma filter that retrieves
   only the newest message, and re-answer the decision question.
3. **Scale to Enron.** Drop a real inbox under
   `Data/SD-04-email/enron/allen-p/inbox/` so `ENRON_AVAILABLE`
   becomes `True`, re-run the pipeline on a real 250-message thread, then ask
   who escalated a topic across several messages.
